In [294]:
import os
import time
import openai
import tiktoken

from tqdm import tqdm
from batches import *
from google import genai
from openai import OpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from IPython.display import clear_output

In [295]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [296]:
load_dotenv()

OPENAI_MODEL: str = "gpt-4.1-mini"

openai_key: str | None = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("OPENAI_API_KEY is not set in environment variables.")

openai_client: OpenAI = OpenAI(api_key=openai_key)

In [297]:
with open("../data/prompts.json", "r", encoding="utf-8") as f:
    prompts: dict[str, str] = json.load(f)

In [298]:
class Chunk(TypedDict):
    uuid: str
    chunk: str
    path: str
    type: str

with open("../data/fixed_size_chunks.json", "r", encoding="utf-8") as f:
    all_chunks: list[Chunk] = json.load(f)

seen: set[str] = set()
unique_chunks: list[Chunk] = []
for obj in all_chunks:
    if obj["uuid"] not in seen:
        seen.add(obj["uuid"])
        unique_chunks.append(obj)

In [299]:
SCHEME: str = "o200k_base"
enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

total_tokens: int = sum(len(enc.encode(obj["chunk"])) for obj in unique_chunks)
print(f"{total_tokens:,}")

18,906,586


In [300]:
class TextContent(TypedDict):
    type: Literal["text"]
    text: str


class Message(TypedDict):
    role: Literal["system", "user"]
    content: str | list[TextContent]


class RequestBody(TypedDict):
    model: str
    messages: list[Message]
    stream: Literal[False]


class BatchRequest(TypedDict):
    custom_id: str
    method: Literal["POST"]
    url: Literal["/v1/chat/completions"]
    body: RequestBody


def create_batch_request(custom_id: str, text: str, system: str, user: str) -> BatchRequest:
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": OPENAI_MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": [{"type": "text", "text": f"{user}{text}"}]},
            ],
            "stream": False,
        },
    }

In [301]:
def count_request_tokens(payload: BatchRequest) -> int:
    total = 0
    for message in payload["body"]["messages"]:
        content = message["content"]
        if isinstance(content, str):
            total += len(enc.encode(content))
        elif isinstance(content, list):
            for block in content:
                if block.get("type") == "text":
                    total += len(enc.encode(block["text"]))
    return total

In [302]:
def wait_for_batch(
    client: OpenAI,
    batch_id: str,
    poll_interval: int = 30,
) -> openai.types.batch.Batch:
    while True:
        batch = client.batches.retrieve(batch_id)
        completed = batch.request_counts.completed
        total = batch.request_counts.total

        clear_output(wait=True)

        if batch.status == "completed":
            print(f"Batch {batch_id} completed! ({completed}/{total})")
            return batch
        elif batch.status in ("failed", "expired", "cancelled"):
            raise RuntimeError(f"Batch {batch_id} ended with status: {batch.status}")

        print(f"[{batch.status}] {completed}/{total} requests done — next check in {poll_interval}s...")
        time.sleep(poll_interval)

In [303]:
all_requests: list[BatchRequest] = [
    create_batch_request(
        custom_id=chunk["uuid"],
        text=chunk["chunk"],
        system=prompts["leaf-chunking"]["system"],
        user=prompts["leaf-chunking"]["user"],
    )
    for chunk in unique_chunks
]

token_counts: list[int] = [count_request_tokens(req) for req in tqdm(all_requests)]

100%|██████████| 37239/37239 [00:09<00:00, 3864.73it/s]


In [304]:
import re

leaf_dir: Path = Path(os.getcwd()).parent / "data/leaf_batches"
leaf_dir.mkdir(parents=True, exist_ok=True)

# Collect all custom_ids already processed in existing batch files
processed_ids: set[str] = set()
for batch_file in leaf_dir.glob("leaf_batch_*.jsonl"):
    if re.match(r"leaf_batch_\d+\.jsonl", batch_file.name):
        with open(batch_file, "r", encoding="utf-8") as f:
            for line in f:
                entry = json.loads(line)
                processed_ids.add(entry["custom_id"])

# Filter to only unprocessed requests
pending_requests: list[BatchRequest] = [
    req for req in all_requests if req["custom_id"] not in processed_ids
]
pending_token_counts: list[int] = [
    token_counts[i] for i, req in enumerate(all_requests)
    if req["custom_id"] not in processed_ids
]

print(f"Total requests    : {len(all_requests):,}")
print(f"Already processed : {len(processed_ids):,}")
print(f"Pending           : {len(pending_requests):,}")

Total requests    : 37,239
Already processed : 14,689
Pending           : 22,550


In [305]:
all_requests = pending_requests
token_counts = pending_token_counts

In [306]:
MAX_TOKENS_PER_BATCH: int = 2_000_000  # GPT-4.1-mini batch limit

batches: list[list[BatchRequest]] = []
current_batch: list[BatchRequest] = []
current_tokens: int = 0

for req, req_tokens in zip(all_requests, token_counts):
    if current_tokens + req_tokens > MAX_TOKENS_PER_BATCH and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_tokens = 0
    current_batch.append(req)
    current_tokens += req_tokens

if current_batch:
    batches.append(current_batch)

print(f"Total requests : {len(all_requests):,}")
print(f"Total batches  : {len(batches):,}")
for i, batch in enumerate(batches):
    batch_tokens = sum(token_counts[all_requests.index(req)] for req in batch)
    print(f"  Batch {i+1}: {len(batch):,} requests | {batch_tokens:,} tokens")

Total requests : 22,550
Total batches  : 10
  Batch 1: 2,449 requests | 1,999,972 tokens
  Batch 2: 2,448 requests | 1,999,187 tokens
  Batch 3: 2,448 requests | 1,999,734 tokens
  Batch 4: 2,448 requests | 1,999,404 tokens
  Batch 5: 2,581 requests | 1,999,482 tokens
  Batch 6: 2,455 requests | 1,999,842 tokens
  Batch 7: 2,454 requests | 1,999,612 tokens
  Batch 8: 2,456 requests | 1,999,797 tokens
  Batch 9: 2,466 requests | 1,999,864 tokens
  Batch 10: 345 requests | 269,726 tokens


In [309]:
for f in leaf_dir.glob("leaf_batch_*.jsonl"):
    f.unlink()

batch_jobs: list[openai.types.Batch] = []

for idx, batch in enumerate(tqdm(batches)):
    file_name: str = f"leaf_batch_{idx}.jsonl"

    with open(leaf_dir / file_name, "w", encoding="utf-8") as f:
        for request in batch:
            f.write(json.dumps(request) + "\n")

    with open(leaf_dir / file_name, "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    batch_job = openai_client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    batch_jobs.append(batch_job)
    print(f"Batch {idx + 1}/{len(batches)} submitted (id: {batch_job.id})")

 10%|█         | 1/10 [00:04<00:38,  4.28s/it]

Batch 1/10 submitted (id: batch_69ab9aa15c448190babfb43a2f8c08be)


 20%|██        | 2/10 [00:07<00:31,  3.92s/it]

Batch 2/10 submitted (id: batch_69ab9aa5080c8190bfb474041d91e2ab)


 30%|███       | 3/10 [00:12<00:28,  4.11s/it]

...

Batch 8/10 submitted (id: batch_69ab9abe9f1881909ed353e8a5a0b16a)


 90%|█████████ | 9/10 [00:38<00:04,  4.49s/it]

Batch 9/10 submitted (id: batch_69ab9ac347688190b765c47e4c4bbde4)


100%|██████████| 10/10 [00:39<00:00,  3.98s/it]

Batch 10/10 submitted (id: batch_69ab9ac4c69c8190aecdd59c2dd96db8)


In [310]:
for idx, batch_job in enumerate(tqdm(batch_jobs)):
    result = wait_for_batch(openai_client, batch_job.id, poll_interval=30)
    print(f"  -> {result.request_counts.completed} completed | {result.request_counts.failed} failed")

    output = openai_client.files.content(result.output_file_id)
    responses: list[dict] = [json.loads(line) for line in output.text.strip().split("\n")]

    with open(master_file, "r", encoding="utf-8") as f:
        existing: list[dict] = json.load(f)

    for response in responses:
        res_custom_id: str = response["custom_id"]
        res_content: str = response["response"]["body"]["choices"][0]["message"]["content"]
        leaves: list[str] = [leaf.strip() for leaf in res_content.split("<|>") if leaf.strip()]
        for i, leaf in enumerate(leaves, start=1):
            existing.append({"uuid": f"{res_custom_id}.{i}", "text": leaf})

    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    print(f"  -> Master file updated ({len(existing)} total leaves): {master_file}")

 10%|█         | 1/10 [04:02<36:26, 242.91s/it]


RuntimeError: Batch batch_69ab9aa5080c8190bfb474041d91e2ab ended with status: failed